# GitHub-Versuchsarchive nach Hugging Face übertragen

**CPU-Laufzeit verwenden; keine GPU erforderlich.** Dieses Notebook startet keine
Hugging-Face-Compute-Umgebung und ist unabhängig von PR #4 nutzbar.

Es bewahrt die historische Auswahl aus GitHub-Commit
`40790e3addb62b5d9e1a9544b67bc4145a3e2f1c`: **491 ursprüngliche Pfade**, gespeichert
als **251 Dateien** plus Zuordnung von **240 identischen Kopien**, etwa **33,6 MB**.
Ziel ist ausschließlich
[`Carlo0001/neura-lara5-cube-stacking-sim-v1`](https://huggingface.co/datasets/Carlo0001/neura-lara5-cube-stacking-sim-v1).
Code, aktuelle Prompts, Konfigurationen und Tests bleiben auf GitHub. Dieses
Notebook löscht keine Quelldateien und verändert keine Git-Historie.


## 1. Quelle laden und prüfen

Die erste Zelle prüft den festgelegten Programm-/Planstand und alle ausgewählten
Dateien aus dem ursprünglichen GitHub-Archiv. Sie braucht keine Zugangsdaten und
lädt noch nichts nach Hugging Face hoch. Bei einem abgebrochenen Quelldownload
eine neue Colab-Laufzeit verwenden oder `TRANSFER` auf einen neuen Ordner setzen.


In [ ]:
import hashlib, json, subprocess, sys, urllib.request
from pathlib import Path

CODE_REF = "f999232af390c5f5eff46b3d4e2ff0ff34966d71"
WORK = Path("/content/lara-evidence-migration")
TRANSFER = WORK/"transfer"
CHECKSUMS = {'tools/migrate_archives.py': '9627272e5775685efc91240a5769988471c9ac741d7eafe7f1bca2aa3f97dbbb', 'config/evidence-migration.json': 'ec70540ce0084d2cb1a1b6bb0e6ca569a9521e7c25bb19a5c68a92946d630e00', 'config/evidence-dataset-card.md': '19b43543f36c73a74df2465eaf489fdf3a900c98332fb476483bc58243e918e4'}
BASE_URL = "https://raw.githubusercontent.com/carlo2004/Simulation-Neura-LARA-5/"+CODE_REF+"/"

for name, expected in CHECKSUMS.items():
    with urllib.request.urlopen(BASE_URL+name, timeout=60) as response:
        content = response.read()
    if hashlib.sha256(content).hexdigest() != expected:
        raise RuntimeError("Programm-/Plan-Prüfsumme stimmt nicht: "+name)
    target = WORK/name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)

SCRIPT = WORK/"tools/migrate_archives.py"
PLAN = WORK/"config/evidence-migration.json"
CARD = WORK/"config/evidence-dataset-card.md"
subprocess.run([sys.executable, str(SCRIPT), "--plan", str(PLAN), "--output", str(TRANSFER)], check=True)
print("Vorbereitung abgeschlossen. Bisher kein HF-Upload und keine GitHub-Löschung.")


## 2. Hochladen und erneut prüfen

Falls nötig unter [Hugging Face → Access Tokens](https://huggingface.co/settings/tokens)
einen **Fine-grained** Token mit Lese-/Schreibrecht nur für das oben genannte
Dataset anlegen. Die folgende Zelle nutzt ein freigegebenes Colab-Secret `HF_TOKEN`
oder fragt den Wert über ein verborgenes Eingabefeld ab. **Nicht in den Chat oder
in den Notebookcode eintragen.** Die Chat-Verbindung ist davon unabhängig.

Nach dem atomaren Upload lädt das Skript jede Datei erneut aus dem festen
Hub-Commit herunter und prüft Größe, SHA-256 und ursprünglichen Git-Blob-Hash.
Erst danach werden die Prüfbestätigung und der zitierbare HF-Link ausgegeben.
Vorhandene abweichende Migrationsdaten und eine vorhandene Datasetkarte werden
nicht überschrieben. Bei einem Uploadfehler bleiben die Quelldateien erhalten.


In [ ]:
import os
from getpass import getpass

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "huggingface_hub==1.33.0"], check=True)
access_token = None
try:
    from google.colab import userdata
    access_token = userdata.get("HF_TOKEN")
except Exception:
    pass
if not access_token:
    access_token = getpass("HF-Token mit Schreibzugriff nur für das Dataset (verborgene Eingabe): ")
if not access_token or not access_token.strip():
    raise ValueError("Kein Token eingegeben; es wird nichts hochgeladen.")

upload_environment = dict(os.environ)
upload_environment["HF_TOKEN"] = access_token.strip()
upload_environment["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
del access_token
try:
    subprocess.run([sys.executable, str(SCRIPT), "--plan", str(PLAN), "--output", str(TRANSFER),
                    "--card", str(CARD), "--upload"], env=upload_environment, check=True)
finally:
    upload_environment.pop("HF_TOKEN", None)

receipt = json.loads((TRANSFER/"migration-receipt.json").read_text())
print("ERFOLGREICH: Alle", receipt["canonical_files_verified"], "Dateien erneut geladen und geprüft.")
print("HF-Link:", receipt["url"])
print("Die ursprünglichen GitHub-Dateien bleiben bis zu einem eigenen Bereinigungs-PR erhalten.")


## Danach

Nach **ERFOLGREICH** den ausgegebenen HF-Link im Chat schicken oder dort „fertig“
schreiben. Dann kann die Übertragung unabhängig nachgeprüft und die GitHub-
Bereinigung als eigener PR vorbereitet werden. Anschließend die Colab-Laufzeit
beenden. Ein eigens angelegter temporärer Token kann danach wieder gelöscht werden.
